In [ ]:
#Deps + Gemma auth. Gemma 3 needs a recent transformers and is GATED on HuggingFace.
!pip -q install -U "transformers>=4.50" "trl>=0.9" peft bitsandbytes datasets accelerate
!pip -q uninstall -y torchao   # Kaggle ships an old torchao that recent peft rejects
import os
# Gemma is gated: accept the licence at https://huggingface.co/google/gemma-3-4b-it, then give a token.
try:                                  # Kaggle: Add-ons -> Secrets -> add HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
# Colab / other: uncomment and paste, or set the HF_TOKEN env var another way:
# os.environ["HF_TOKEN"] = "hf_xxx"
from huggingface_hub import login
if os.environ.get("HF_TOKEN"): login(os.environ["HF_TOKEN"])
print("deps installed; HF auth set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
#Clone the repo (prepared Jira data + regression probes) + config + helpers. Works on Kaggle & Colab.
import os, json, re, random
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
OUT  = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO = f"{OUT}/repo"
if not os.path.isdir(REPO):
    os.system(f"git clone --depth 1 https://github.com/ianrussel/llm-finetuning-lab.git {REPO}")
ROOT   = f"{REPO}/trainings"
TASK   = f"{ROOT}/task-support-tickets/task2/data"               # prepared Jira gold/train/labels (committed)
PROBES = f"{ROOT}/task-dataset-generation-pipeline1/eval"        # regression probes (reused)

CFG = {
    "name": "jira-issue-type",
    "base_model": "microsoft/Phi-4-mini-instruct",   # MIT, ungated, text-only (no HF token needed)
    "lora": {"rank": 16, "alpha": 32, "dropout": 0.05, "target_modules": "all-linear"},
    "train": {"epochs": 6, "lr": 2e-4, "max_len": 1024, "batch": 1, "grad_accum": 16, "seeds": [0, 1],
              "assistant_only_loss": False,   # Phi-4-mini chat template lacks {% generation %}; train full-seq
              # adaptive length ON via eval_mode: resident -- scores the model already in memory
              # (no 2nd bf16 copy), so it fits a 3.8B on a T4. Resident is the as-trained 4-bit model
              # (a best-epoch/regression proxy); the bf16 gate after training is the final arbiter.
              "early_stopping": {"enabled": False, "eval_mode": "resident", "patience": 2,  # OFF: 3.8B on a T4 (clean OOMs, resident generate() errors); fixed-epoch + gate is reliable
                                 "min_task_delta": 0.005, "regression_tolerance": 0.05,
                                 "eval_task_limit": 0, "eval_batch": 8}},
    "guardrails": {"replay_mix": True},
    "acceptance": {"min_task_gain_macro_f1": 0.05, "max_regression_drop": 0.05},
    "adjust_on_fail": {"force_replay": True, "lower_lr_factor": 0.5, "reduce_epochs_to": 2},
}
LABELS = [l.strip() for l in open(f"{TASK}/labels.txt") if l.strip()]

def read_jsonl(p): return [json.loads(l) for l in open(p) if l.strip()]
def normalize(s): return " ".join(str(s).lower().split())
def c_predict_label(out, labels):
    tail = out.rsplit("</think>", 1)[-1] if "</think>" in out else out
    by = {normalize(l): l for l in labels}
    lines = [x for x in tail.splitlines() if x.strip()]
    last = normalize(lines[-1]) if lines else ""
    if last in by: return by[last]
    low = normalize(tail); hits = [l for l in labels if normalize(l) in low]
    return max(hits, key=len) if hits else None
def macro_f1(gold, pred, labels):
    tot = 0.0
    for l in labels:
        tp = sum(1 for g,p in zip(gold,pred) if g==l and p==l)
        fp = sum(1 for g,p in zip(gold,pred) if g!=l and p==l)
        fn = sum(1 for g,p in zip(gold,pred) if g==l and p!=l)
        pr = tp/(tp+fp) if tp+fp else 0.0; rc = tp/(tp+fn) if tp+fn else 0.0
        tot += 2*pr*rc/(pr+rc) if pr+rc else 0.0
    return tot/len(labels)
def score_probes(raw, probes, mode):
    ok = 0
    for o, p in zip(raw, probes):
        want = [a.lower() for a in p["answers"]]; low = o.lower()
        ok += (all(a in low for a in want) if mode=="all" else any(a in low for a in want))
    return ok/len(probes) if probes else 0.0
print("ready | labels:", LABELS, "| gold:", len(read_jsonl(f"{TASK}/gold.jsonl")), "| train_mix:", len(read_jsonl(f"{TASK}/train_mix.jsonl")))

In [ ]:
#Config-driven training (adaptive length: two-axis early stopping; eval_mode clean|resident)
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

_PROBES = [("sentinel","sentinel.jsonl","any",128),
           ("reasoning","reasoning_probes.jsonl","any",256),
           ("tools","tool_probes.jsonl","all",128)]

class TwoAxisEarlyStopping(TrainerCallback):
    def __init__(self, tok, out, es):
        self.tok, self.out = tok, out
        self.mode = es.get("eval_mode", "clean")
        self.patience = es.get("patience", 2); self.min_delta = es.get("min_task_delta", 0.005)
        self.tol = es.get("regression_tolerance", CFG["acceptance"]["max_regression_drop"])
        self.batch = es.get("eval_batch", 8); lim = es.get("eval_task_limit", 0)
        gold = read_jsonl(f"{TASK}/gold.jsonl"); self.gold = gold[:lim] if lim else gold
        self.probes = []
        for axis, fn, mode, mx in _PROBES:
            p = f"{PROBES}/{fn}"
            if os.path.exists(p):
                rws = read_jsonl(p)
                self.probes.append((axis, [[{"role":"user","content":r["question"]}] for r in rws], rws, mode, mx))
        self.history, self.peaks = [], {}
        self.best_task, self.best_epoch, self.bad, self.stop_reason = float("-inf"), None, 0, None
    def _run(self, m, tok):
        raw = _gen(m, tok, [r["messages"] for r in self.gold], 384, self.batch)
        task = macro_f1([r["label"] for r in self.gold], [c_predict_label(o, LABELS) for o in raw], LABELS)
        sc = {axis: score_probes(_gen(m, tok, pr, mx, self.batch), rws, mode) for axis, pr, rws, mode, mx in self.probes}
        return task, sc
    def _score_clean(self, adapter_dir):
        m, tok = _load(adapter_dir)
        try: return self._run(m, tok)
        finally: del m; torch.cuda.empty_cache()
    def _score_resident(self, model):
        pad = self.tok.padding_side; cache = getattr(model.config, "use_cache", True); training = model.training
        gc = bool(getattr(model, "is_gradient_checkpointing", False))
        self.tok.padding_side = "left"; model.config.use_cache = True; model.eval()
        if gc:
            try: model.gradient_checkpointing_disable()
            except Exception: pass
        try: return self._run(model, self.tok)
        finally:
            self.tok.padding_side = pad; model.config.use_cache = cache
            if gc:
                try: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
                except Exception: pass
            if training: model.train()
    def on_epoch_end(self, args, state, control, **kw):
        model = kw.get("model")
        if model is None: return control
        epoch = round(state.epoch or 0)
        try:
            if self.mode == "resident":
                task, sc = self._score_resident(model)
            else:
                tmp = f"{self.out}/_epoch_eval"; model.save_pretrained(tmp); task, sc = self._score_clean(tmp)
        except Exception as e:
            print(f"[early-stop] epoch {epoch}: eval skipped ({type(e).__name__}: {e})"); return control
        reg_drop = max([self.peaks.get(a,v)-v for a,v in sc.items()] or [0.0]); reg_ok = reg_drop <= self.tol + 1e-9
        self.history.append({"epoch":epoch,"task":round(task,4),**{k:round(v,4) for k,v in sc.items()}})
        print(f"[early-stop] epoch {epoch} ({self.mode}): task={task:.3f} "+" ".join(f"{k}={v:.3f}" for k,v in sc.items())+f" | reg_drop_from_peak={reg_drop:.3f} (tol {self.tol:.3f})")
        if reg_ok and task > self.best_task + self.min_delta:
            self.best_task, self.best_epoch, self.bad = task, epoch, 0
            model.save_pretrained(self.out); self.tok.save_pretrained(self.out)
            print(f"[early-stop] new best task={task:.3f}; kept adapter from epoch {epoch}")
        else: self.bad += 1
        for a, v in sc.items(): self.peaks[a] = max(self.peaks.get(a, v), v)
        if not reg_ok: self.stop_reason = f"regression axis fell {reg_drop:.3f} > tol {self.tol:.3f} at epoch {epoch}"
        elif self.bad >= self.patience: self.stop_reason = f"task macro-F1 plateaued ({self.bad} eval(s)) at epoch {epoch}"
        if self.stop_reason:
            print(f"[early-stop] stopping: {self.stop_reason}; best epoch={self.best_epoch}"); control.should_training_stop = True
        return control

def train_one(data_file, run_name, lr=None, epochs=None, seed=0):
    t, lo = CFG["train"], CFG["lora"]
    lr = t["lr"] if lr is None else lr; epochs = t["epochs"] if epochs is None else epochs
    es = t.get("early_stopping", {}); es_on = bool(es.get("enabled", False))
    tok = AutoTokenizer.from_pretrained(CFG["base_model"])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(CFG["base_model"], quantization_config=quant,
        torch_dtype=torch.bfloat16, device_map="auto"); model.config.use_cache = False
    lora = LoraConfig(r=lo["rank"], lora_alpha=lo["alpha"], lora_dropout=lo["dropout"],
        bias="none", task_type="CAUSAL_LM", target_modules=lo["target_modules"])
    ds = load_dataset("json", data_files=f"{TASK}/{data_file}", split="train")
    out = f"{OUT}/{run_name}"
    cfg = SFTConfig(output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=t["batch"],
        gradient_accumulation_steps=t["grad_accum"], learning_rate=lr, lr_scheduler_type="cosine",
        warmup_ratio=0.05, logging_steps=10, save_strategy=("no" if es_on else "epoch"), bf16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=t["max_len"], packing=False, assistant_only_loss=t.get("assistant_only_loss", True), seed=seed, report_to="none")
    stopper = TwoAxisEarlyStopping(tok, out, es) if es_on else None
    if es_on:
        print(f"[train] {run_name}: adaptive length on (mode={stopper.mode}, max epochs={epochs}, patience={stopper.patience}, reg_tol={stopper.tol})")
    tr = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora, processing_class=tok,
                    callbacks=[stopper] if stopper else [])
    r = tr.train()
    if not (stopper and stopper.best_epoch is not None): tr.save_model(out); tok.save_pretrained(out)
    tail = (f" best_epoch={stopper.best_epoch}/{epochs} stop={stopper.stop_reason or 'reached max epochs'}" if stopper else "")
    print(f"[train] {run_name}: loss={r.training_loss:.4f}{tail} (data={data_file}, lr={lr}, seed={seed}, max_epochs={epochs})")
    return out

In [ ]:
#Two-axis evaluation (task gold from TASK, regression probes from PROBES; batch matches early stop)
def _load(adapter=None):
    tok=AutoTokenizer.from_pretrained(CFG["base_model"]); tok.padding_side="left"
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    m=AutoModelForCausalLM.from_pretrained(CFG["base_model"], torch_dtype=torch.bfloat16)
    if adapter:
        from peft import PeftModel; m=PeftModel.from_pretrained(m, adapter)
    return m.to("cuda").eval(), tok

@torch.no_grad()
def _gen(m, tok, prompts, max_new, batch=8):
    outs=[]
    for i in range(0,len(prompts),batch):
        ch=prompts[i:i+batch]
        txt=[tok.apply_chat_template(x, add_generation_prompt=True, tokenize=False) for x in ch]
        enc=tok(txt, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        g=m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(ch)): outs.append(tok.decode(g[j][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return outs

def _probe(m, tok, fname, mode, max_new, batch=8):
    probes=read_jsonl(f"{PROBES}/{fname}")
    raw=_gen(m, tok, [[{"role":"user","content":p["question"]}] for p in probes], max_new, batch)
    return {"n":len(probes), "score":score_probes(raw, probes, mode)}

def evaluate(name, adapter=None):
    gold=read_jsonl(f"{TASK}/gold.jsonl"); eb=CFG["train"]["early_stopping"].get("eval_batch",8)
    m,tok=_load(adapter)
    raw=_gen(m, tok, [r["messages"] for r in gold], 384, eb)
    g=[r["label"] for r in gold]; p=[c_predict_label(o,LABELS) for o in raw]
    task={"macro_f1":macro_f1(g,p,LABELS), "accuracy":sum(1 for a,b in zip(g,p) if a==b)/len(g),
          "valid":sum(1 for x in p if x is not None)/len(g)}
    res={"name":name, "task":task,
         "sentinel":_probe(m,tok,"sentinel.jsonl","any",128,eb),
         "reasoning":_probe(m,tok,"reasoning_probes.jsonl","any",256,eb),
         "tools":_probe(m,tok,"tool_probes.jsonl","all",128,eb)}
    json.dump(res, open(f"{OUT}/result_{name}.json","w"), indent=2)
    print(f"[eval] {name}: macroF1={task['macro_f1']:.3f} sentinel={res['sentinel']['score']:.3f} "
          f"reasoning={res['reasoning']['score']:.3f} tools={res['tools']['score']:.3f}")
    del m; torch.cuda.empty_cache(); return res

In [ ]:
#The gate + the orchestrated loop (median across seeds)
import statistics
def gate(base, cand):
    a=CFG["acceptance"]; task_gain=cand["task"]["macro_f1"]-base["task"]["macro_f1"]
    worst=max((base[k]["score"]-cand[k]["score"]) for k in ["sentinel","reasoning","tools"])
    accept=task_gain>=a["min_task_gain_macro_f1"] and worst<=a["max_regression_drop"]
    print(f"[gate] task_gain={task_gain:+.3f} (min {a['min_task_gain_macro_f1']:+.3f}) | worst_drop={worst:.3f} "
          f"(max {a['max_regression_drop']:.3f}) -> {'ACCEPT' if accept else 'REJECT'}")
    return accept, task_gain, worst
def run_seeds(base, base_name, data_file, lr=None, epochs=None):
    seeds=CFG["train"].get("seeds",[0]); runs=[]
    for s in seeds:
        name_s=f"{base_name}-s{s}" if len(seeds)>1 else base_name
        adapter=train_one(data_file, name_s, lr=lr, epochs=epochs, seed=s)
        cand=evaluate(name_s, adapter); acc,gain,drop=gate(base, cand)
        runs.append({"seed":s,"name":name_s,"adapter":adapter,"accept":acc,"gain":gain,"drop":drop})
    return runs
def aggregate(runs):
    a=CFG["acceptance"]; gains=[r["gain"] for r in runs]; drops=[r["drop"] for r in runs]
    mg,md=statistics.median(gains),statistics.median(drops)
    accept=mg>=a["min_task_gain_macro_f1"] and md<=a["max_regression_drop"]
    passing=[r for r in runs if r["accept"]]; chosen=max(passing or runs, key=lambda r: r["gain"])
    print(f"[gate] across seeds: {'ACCEPT' if accept else 'REJECT'} (median gain={mg:+.3f}, median drop={md:.3f}, "
          f"passed={len(passing)}/{len(runs)}, range=[{min(gains):+.3f},{max(gains):+.3f}]) -> chosen {chosen['name']}")
    return accept, chosen
base=evaluate("base")
replay=CFG["guardrails"]["replay_mix"]
name1=f"{CFG['name']}-{'replay' if replay else 'noreplay'}"
runs1=run_seeds(base, name1, "train_mix.jsonl" if replay else "train_synth.jsonl")
ok1,chosen1=aggregate(runs1); final=chosen1["adapter"] if ok1 else None
if not ok1:
    adj=CFG["adjust_on_fail"]; lr=CFG["train"]["lr"]*adj["lower_lr_factor"]
    print(f"\n[pipeline] gate rejected; adjusted re-run (lr={lr}, epochs={adj['reduce_epochs_to']})")
    runs2=run_seeds(base, f"{CFG['name']}-adj", "train_mix.jsonl", lr=lr, epochs=adj["reduce_epochs_to"])
    ok2,chosen2=aggregate(runs2); final=chosen2["adapter"] if ok2 else None
print("\nACCEPTED:", final or "none")

In [ ]:
#Third axis: knowledge absorption (closed-book, reported not gated). Needs knowledge_probes.jsonl
# (committed in task2/data, present after the repo clone) + the accepted adapter `final` from above.
kp = read_jsonl(f"{TASK}/knowledge_probes.jsonl")
def knowledge_score(adapter):
    m, tok = _load(adapter)
    raw = _gen(m, tok, [[{"role":"user","content":p["question"]}] for p in kp], 64,
               CFG["train"]["early_stopping"].get("eval_batch", 8))
    s = score_probes(raw, kp, "any"); del m; torch.cuda.empty_cache(); return s
kb_base = knowledge_score(None)
if final:
    kb_ft = knowledge_score(final); gain = kb_ft - kb_base
    verdict = "some domain knowledge stuck" if gain >= 0.02 else "little/no shift (expected for a small fine-tune)"
    print(f"[knowledge] {len(kp)} closed-book probes | base={kb_base:.3f} fine-tuned={kb_ft:.3f} gain={gain:+.3f} -> {verdict}")
    json.dump({"n":len(kp),"base":round(kb_base,4),"fine_tuned":round(kb_ft,4),"knowledge_gain":round(gain,4)},
              open(f"{OUT}/knowledge.json","w"), indent=2)
else:
    print(f"[knowledge] no accepted adapter; base only = {kb_base:.3f}")

In [ ]:
#Download adapters + results
import shutil, glob
made=[]
for d in sorted(glob.glob(f"{OUT}/*/")):
    if os.path.exists(os.path.join(d,"adapter_config.json")):
        b=d.rstrip("/"); shutil.make_archive(b,"zip",b); made.append(os.path.basename(b))
print("zipped adapters:", made)
print("Download the lora *.zip and result_*.json from the Output tab.")